# Born-projected angular coefficients from POWHEG LHE events

This notebook builds the first complete analysis chain for off-shell $H\to ZZ\to e^+e^-\mu^+\mu^-$ events:

1. read the LHE file with `pylhe`;
2. retain the Higgs and intermediate $Z$ records when present, the four charged leptons, and final-state partons;
3. remove the color-singlet transverse recoil with the Born projection of [arXiv:2606.11083](https://arxiv.org/abs/2606.11083);
4. construct the angular coordinates and standard four-lepton observables, following the terminology of [arXiv:1208.4018](https://arxiv.org/abs/1208.4018);
5. train a reweighted density-ratio estimator with [`nsbi-common-utils`](https://github.com/iris-hep/nsbi-lhc-toolkit); and
6. compare the learned differential angular coefficient with the direct weighted projection.

Our fixed off-shell convention is $Z_1=Z_{\mu\mu}$ and $Z_2=Z_{ee}$; there is no mass ordering. The harmonic coordinates use the **positively charged** lepton in each $Z$ rest frame. The separately named `theta1_standard` and `theta2_standard` use the negatively charged leptons, as in arXiv:1208.4018.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from offshell_angles import (
    angular_target,
    load_lhe_dataframe,
    prepare_weighted_classification,
    recover_conditional_moment,
)

hep.style.use("ATLAS")
pd.set_option("display.max_columns", 80)
RNG_SEED = 20260718

## 1. Read the local LHE file

Set `LHE_FILE` to a file visible inside the JupyterLab pod. Both plain `.lhe` and gzip-compressed `.lhe.gz` files are supported by `pylhe`. During development, set `MAX_EVENTS` to a small integer; use `None` for the full sample.

With `STRICT_TOPOLOGY=True`, the reader stops unless every accepted event has exactly one final-state $e^-$, $e^+$, $\mu^-$, and $\mu^+$. Set it to `False` only when intentionally filtering a mixed-topology LHE file.

In [ ]:
LHE_FILE = Path("/path/visible/from/jupyter/events.lhe")
MAX_EVENTS = 20_000  # None for the complete file
STRICT_TOPOLOGY = True

if not LHE_FILE.is_file():
    raise FileNotFoundError(
        f"Update LHE_FILE: the configured path does not exist: {LHE_FILE}"
    )

events = load_lhe_dataframe(
    LHE_FILE,
    max_events=MAX_EVENTS,
    strict=STRICT_TOPOLOGY,
    include_momenta=False,
)
print(f"Loaded {len(events):,} selected events")
events.head()

In [ ]:
object_summary = {
    "events": len(events),
    "sum_weights": events["weight"].sum(),
    "negative_weight_fraction": (events["weight"] < 0).mean(),
    "events_with_higgs_record": (events["n_higgs_lhe"] > 0).sum(),
    "events_with_two_z_records": (events["n_z_lhe"] >= 2).sum(),
    "events_with_final_parton": (events["n_final_partons"] > 0).sum(),
}
pd.Series(object_summary, name="value")

## 2. Born-like projection

Let $k=\sum_{i=1}^4p_{\ell_i}$ be the measured four-lepton momentum. The projection applies the same Lorentz transformation to every lepton:

$$p_i^{\mathrm B}=B_L^{-1}B_TB_Lp_i.$$

$B_L$ first removes the longitudinal momentum of $k$, $B_T$ then removes the transverse momentum of the longitudinally boosted system, and $B_L^{-1}$ restores the original color-singlet rapidity. Consequently,

$$m_{4\ell}^{\mathrm B}=m_{4\ell},\qquad y_{4\ell}^{\mathrm B}=y_{4\ell},\qquad p_{T,4\ell}^{\mathrm B}=0,$$

and every internal invariant mass, including $m_{\mu\mu}$ and $m_{ee}$, is unchanged. The following checks should be at floating-point precision.

In [ ]:
projection_checks = pd.DataFrame({
    "abs_delta_m4l": np.abs(events["born_m4l"] - events["raw_m4l"]),
    "abs_delta_y4l": np.abs(events["born_y4l"] - events["raw_y4l"]),
    "born_pt4l": events["born_pt4l"],
})
display(projection_checks.describe(percentiles=[0.5, 0.9, 0.99]).T)
assert np.allclose(events["born_m4l"], events["raw_m4l"], rtol=2e-10, atol=2e-10)
assert np.allclose(events["born_y4l"], events["raw_y4l"], rtol=2e-10, atol=2e-10)
assert np.max(events["born_pt4l"] / np.maximum(events["raw_m4l"], 1.0)) < 2e-10

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(events["raw_pt4l"], bins=60, weights=events["weight"], histtype="step", label="POWHEG")
axes[0].hist(events["born_pt4l"], bins=60, weights=events["weight"], histtype="step", label="Born projected")
axes[0].set(xlabel=r"$p_{T,4\ell}$ [GeV]", ylabel="weighted events")
axes[0].legend()
axes[1].scatter(events["raw_y4l"], events["born_y4l"] - events["raw_y4l"], s=4, alpha=0.25)
axes[1].set(xlabel=r"raw $y_{4\ell}$", ylabel=r"$y_{4\ell}^{\mathrm B}-y_{4\ell}$")
fig.tight_layout()

## 3. Angular and differential variables

In the Born-projected four-lepton rest frame, the local $\hat z_i$ axis follows $Z_i$. The $\hat y_i$ axis is normal to the beam--$Z_i$ production plane and $\hat x_i=\hat y_i\times\hat z_i$. After boosting the positively charged lepton into its parent-$Z$ rest frame, its direction defines $\Omega_i=(\theta_i,\phi_i)$.

The reader also supplies the usual variables $m_{Z_1}$, $m_{Z_2}$, $m_{ZZ}$, $\cos\theta^*$, $\Phi$, $\Phi_1$, and $\Psi=\Phi_1+\Phi/2$ (wrapped to $[-\pi,\pi)$). Here $Z_1$ is always the dimuon system. At exactly collinear production, an azimuthal origin is mathematically undefined; the record flags this measure-zero case.

In [ ]:
angle_columns = ["cos_theta1", "phi1", "cos_theta2", "phi2", "cos_theta_star", "Phi"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, column in zip(axes.flat, angle_columns):
    axis.hist(events[column].dropna(), bins=50, weights=events.loc[events[column].notna(), "weight"], histtype="step")
    axis.set(xlabel=column, ylabel="weighted events")
fig.tight_layout()

events[["m_Z1", "m_Z2", "m_ZZ", "cos_theta_star", "frame_degenerate", "standard_angles_degenerate"]].describe(include="all")

## 4. Harmonic projection and the $4\pi$ convention

We use

$$p(\Omega_1,\Omega_2,x)=\frac{1}{4\pi}\sum_{\alpha\preceq\beta}\mathcal S_{\alpha\beta}(x)\,\mathcal Y^{(+)}_{\alpha\beta}(\Omega_1,\Omega_2).$$

For an orthonormal basis, multiplication by $\mathcal Y^{(+)*}_{\alpha\beta}$ and integration over both solid angles gives

$$\mathcal S_{\alpha\beta}(x)=4\pi\int d\Omega_1d\Omega_2\;\mathcal Y^{(+)*}_{\alpha\beta}(\Omega_1,\Omega_2)\,p(\Omega_1,\Omega_2,x).$$

In particular, $\mathcal Y^{(+)}_{00;00}=1/(4\pi)$, so angular integration returns $\mathcal S_{00;00}(x)$. A direct MC estimator in an $x$ bin is therefore

$$\widehat{\mathcal S}_{\alpha\beta}(\text{bin})=4\pi\sum_{i\in\text{bin}}w_i\,\mathcal Y^{(+)*}_{\alpha\beta}(\Omega_{1,i},\Omega_{2,i}).$$

For complex harmonics we train the real and imaginary components separately.

In [ ]:
# Select one component of one symmetrized coefficient.
COEFFICIENT = dict(l1=2, m1=0, l2=2, m2=0, component="real")

h, target, harmonic_bound = angular_target(
    events["theta1"].to_numpy(),
    events["phi1"].to_numpy(),
    events["theta2"].to_numpy(),
    events["phi2"].to_numpy(),
    **COEFFICIENT,
)
events = events.assign(h_target=h, t_target=target)
print(f"Analytic harmonic bound M = {harmonic_bound:.8g}")
print(f"Observed h range = [{h.min():.8g}, {h.max():.8g}]")
print(f"Integrated direct coefficient = {4*np.pi*np.sum(events['weight']*events['h_target']):.8g}")

## 5. Why the duplicated weighted samples recover the conditional angular moment

For the chosen real component define $h(\Omega)=\operatorname{Re}\mathcal Y^{(+)*}_{\alpha\beta}(\Omega)$ and choose a rigorous $M$ such that $|h|\le M$. Then

$$t(\Omega)=\frac12\left(1+\frac{h(\Omega)}{M}\right)\in[0,1].$$

Duplicate each MC event into a numerator row with weight $w_it_i$ and a denominator row with weight $w_i(1-t_i)$. At fixed $x$, these two weighted measures are

$$A(x)=\int t(\Omega)p(\Omega,x)d\Omega,\qquad B(x)=\int[1-t(\Omega)]p(\Omega,x)d\Omega.$$

Minimizing the weighted binary cross entropy pointwise gives $s^*(x)=A(x)/[A(x)+B(x)]$ and hence $A/B=s^*/(1-s^*)$. Therefore

$$\mathbb E[h\mid x]=M\left(2\frac{A(x)}{A(x)+B(x)}-1\right).$$

`density_ratio_trainer` requires each class to be normalized independently. If $Z_t=\sum_iw_it_i$ and $Z_{1-t}=\sum_iw_i(1-t_i)$, its score yields the normalized-density ratio $r_{\rm norm}=(A/Z_t)/(B/Z_{1-t})$. We must restore

$$\rho(x)=\frac{A(x)}{B(x)}=\frac{Z_t}{Z_{1-t}}r_{\rm norm}(x),\qquad \mathbb E[h\mid x]=M\frac{\rho(x)-1}{\rho(x)+1}.$$

Finally,

$$\mathcal S_{\alpha\beta}(x)=4\pi\frac{d\sigma}{dx}\,\mathbb E[\mathcal Y^{(+)*}_{\alpha\beta}\mid x].$$

> This BCE/KL construction assumes a non-negative event measure. The helper deliberately rejects negative MC weights; signed NLO weights require a different estimator or a separately validated signed-measure treatment.

In [ ]:
FEATURES = ["m_Z1", "m_Z2", "m_ZZ", "cos_theta_star"]
EVALUATION_FRACTION = 0.25

if (events["weight"] < 0).any():
    raise ValueError(
        "Negative LHE weights were found. Do not apply the BCE construction "
        "without first choosing and validating a signed-weight strategy."
    )
if events[FEATURES + ["t_target"]].isna().any().any():
    raise ValueError("Non-finite training inputs were found; inspect degenerate events first.")

fit_events, evaluation_events = train_test_split(
    events, test_size=EVALUATION_FRACTION, random_state=RNG_SEED
)
fit_events = fit_events.reset_index(drop=True)
evaluation_events = evaluation_events.reset_index(drop=True)

training_dataframe, class_normalizations = prepare_weighted_classification(
    fit_events, fit_events["t_target"].to_numpy(), shuffle_seed=RNG_SEED
)
display(pd.Series({
    "fit_source_events": len(fit_events),
    "duplicated_training_rows": len(training_dataframe),
    "evaluation_events": len(evaluation_events),
    "Z_t": class_normalizations.z_t,
    "Z_1_minus_t": class_normalizations.z_one_minus_t,
    "odds_correction": class_normalizations.odds_correction,
}))
training_dataframe.groupby("train_labels")["weights_normed"].sum()

The original events are split **before** duplication, so `evaluation_events` is statistically independent and should be used for the final closure plot. The toolkit currently performs its own row-level train/holdout split; because the two weighted copies of a source event can land on opposite sides, its built-in holdout diagnostics can be optimistic for this soft-label construction. They are useful for debugging, but the external event-level evaluation is the meaningful closure test.

In [ ]:
from nsbi_common_utils.training import density_ratio_trainer

MODELS_DIR = PROJECT_ROOT / "models" / "angular_ratio"
FIGURES_DIR = PROJECT_ROOT / "figures" / "angular_ratio"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ratio_trainer = density_ratio_trainer(
    dataset=training_dataframe,
    weights=training_dataframe["weights_normed"].to_numpy(),
    training_labels=training_dataframe["train_labels"].to_numpy(),
    features=FEATURES,
    features_scaling=FEATURES,
    sample_name=["t-weighted", "one-minus-t-weighted"],
    output_name="angular_moment",
    path_to_figures=f"{FIGURES_DIR}/",
    path_to_models=f"{MODELS_DIR}/",
    use_log_loss=False,
)

RUN_MODEL = False  # Set True after the input/projection checks look sensible.
LOAD_TRAINED_MODEL = False

if RUN_MODEL:
    ratio_trainer.train(
        hidden_layers=3,
        neurons=128,
        number_of_epochs=150,
        batch_size=1024,
        learning_rate=1.0e-3,
        scalerType="StandardScaler",
        calibration=False,
        callback=True,
        callback_patience=20,
        activation="swish",
        rnd_seed=RNG_SEED,
        validation_split=0.1,
        holdout_split=0.15,
        load_trained_models=LOAD_TRAINED_MODEL,
        num_workers=0,
    )
else:
    print("Training is disabled. Set RUN_MODEL=True and rerun this cell.")

## 6. Recover the moment and close the differential coefficient

The toolkit prediction is a classifier score $s_{\rm norm}$. We first form $r_{\rm norm}=s_{\rm norm}/(1-s_{\rm norm})$, restore the class-normalization factor, and recover $\widehat{\mathbb E[h\mid x]}$. The closure comparison below uses only the untouched event-level evaluation sample.

In [ ]:
if RUN_MODEL:
    from nsbi_common_utils.training import predict_with_model

    score_norm = predict_with_model(
        evaluation_events[FEATURES],
        scaler=ratio_trainer.scaler,
        model=ratio_trainer.model_NN,
        calibration_model=ratio_trainer.histogram_calibrator,
        use_log_loss=ratio_trainer.use_log_loss,
    )
    score_norm = np.clip(np.asarray(score_norm), 1.0e-8, 1.0 - 1.0e-8)
    normalized_ratio = score_norm / (1.0 - score_norm)
    eta_hat, h_hat = recover_conditional_moment(
        normalized_ratio, class_normalizations, harmonic_bound
    )
    evaluation_events = evaluation_events.assign(
        score_norm=score_norm, eta_hat=eta_hat, h_hat=h_hat
    )
    display(evaluation_events[[*FEATURES, "h_target", "h_hat"]].head())
else:
    print("Run the training cell first.")

In [ ]:
if RUN_MODEL:
    mzz = evaluation_events["m_ZZ"].to_numpy()
    bins = np.linspace(np.quantile(mzz, 0.01), np.quantile(mzz, 0.99), 16)
    centers = 0.5 * (bins[1:] + bins[:-1])
    # Undo the random evaluation subsampling in expectation.
    evaluation_weights = (
        evaluation_events["weight"].to_numpy()
        * len(events) / len(evaluation_events)
    )
    direct, _ = np.histogram(
        mzz, bins=bins, weights=4.0*np.pi*evaluation_weights*evaluation_events["h_target"]
    )
    learned, _ = np.histogram(
        mzz, bins=bins, weights=4.0*np.pi*evaluation_weights*evaluation_events["h_hat"]
    )
    direct_variance, _ = np.histogram(
        mzz, bins=bins, weights=(4.0*np.pi*evaluation_weights*evaluation_events["h_target"])**2
    )

    fig, axis = plt.subplots(figsize=(8, 5.5))
    axis.errorbar(centers, direct, yerr=np.sqrt(direct_variance), fmt="o", label="direct angular projection")
    axis.stairs(learned, bins, label="density-ratio conditional moment", linewidth=2)
    axis.axhline(0.0, color="black", linewidth=0.8)
    axis.set(xlabel=r"$m_{ZZ}$ [GeV]", ylabel=r"$\mathcal S_{\alpha\beta}$ per bin")
    axis.legend()
    fig.tight_layout()
else:
    print("Run the training and prediction cells first.")

### Required validation before physics use

Repeat the closure for every real and imaginary component retained in the expansion, vary architecture and random seeds, and check stability versus LHE statistics and binning. Validate angle signs against a small hand-checked sample. For a final result, use event-grouped cross-validation or independently generated samples and propagate finite-MC, training, and model-ensemble uncertainties. The direct points above include only their finite-evaluation-sample weighted variance; the learned curve does not yet include model uncertainty.